# HCX Few-shot 기반 뉴스 통계 주장 추출 및 KOSIS 데이터 추천

`AI_기반_뉴스_사실검증_시스템_프로젝트_데이터_본문전처리.csv`에서 **후보 라벨 파일에 등장하는 기사만** 골라 HCX로 통계 주장을 추출합니다. HCX는 구조화된 주장과 KOSIS 검색어를 만들고, 실제 기관/통계표는 KOSIS 공식 검색 API 결과에서 선택합니다. 마지막에는 `candidate_labeling_pilot_relaxed_team2_codex_prelabel.csv`와 비교해 성능을 계산합니다.

> 주의: 후보 파일은 실버 라벨이며 `gold_source_scope` 등이 미확정인 행이 많습니다. 따라서 집계통계 여부와 claim 문장 매칭은 전체 평가하고, 기관/표 평가는 정답이 채워진 행만 별도 집계합니다.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [25]:
from __future__ import annotations

import ast
import json
import os
import re
import time
import uuid
from difflib import SequenceMatcher
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display

# 사용자 Google Drive 마운트 경로
ROOT_CANDIDATES = [
    Path('/drive/MyDrive/멋사'),          # 사용자 지정 마운트
    Path('/content/drive/MyDrive/멋사'),  # Google Colab 기본 마운트
]
ROOT = next((p for p in ROOT_CANDIDATES if (p / 'data').is_dir()), ROOT_CANDIDATES[0])
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "output" / "fewshot"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ARTICLE_PATH = DATA_DIR / "AI_기반_뉴스_사실검증_시스템_프로젝트_데이터_본문전처리.csv"
GOLD_PATH = DATA_DIR / "candidate_labeling_pilot_relaxed_team2_codex_prelabel.csv"
CACHE_PATH = OUTPUT_DIR / "hcx_claim_cache.jsonl"
RESULT_PATH = OUTPUT_DIR / "fewshot_hcx_predictions.csv"
RECOMMENDATION_PATH = OUTPUT_DIR / "fewshot_kosis_recommendations.csv"

if not DATA_DIR.exists():
    checked = ', '.join(str(p / 'data') for p in ROOT_CANDIDATES)
    raise FileNotFoundError(
        f"데이터 폴더를 찾을 수 없습니다. 확인한 경로: {checked}. "
        "Colab에서는 먼저 drive.mount('/content/drive')를 실행하세요."
    )

load_dotenv(ROOT / ".env")
HCX_API_KEY = os.getenv("NCP_CLOVASTUDIO_API_KEY", "").strip()
KOSIS_API_KEY = os.getenv("KOSIS_API_KEY", "").strip()

# HCX-005/HCX-DASH-002는 Chat Completions v3를 사용합니다.
HCX_MODEL = os.getenv("HCX_MODEL", "HCX-005")
HCX_URL = f"https://clovastudio.stream.ntruss.com/v3/chat-completions/{HCX_MODEL}"
KOSIS_SEARCH_URL = "https://kosis.kr/openapi/statisticsSearch.do"

DRY_RUN = True       # API를 실제 호출하려면 False
MAX_ARTICLES = None  # 빠른 시험은 3 등으로 지정
TOP_K_KOSIS = 5
RANDOM_SEED = 42

print({
    "HCX key loaded": bool(HCX_API_KEY),
    "KOSIS key loaded": bool(KOSIS_API_KEY),
    "model": HCX_MODEL,
    "dry_run": DRY_RUN,
})


{'HCX key loaded': True, 'KOSIS key loaded': True, 'model': 'HCX-005', 'dry_run': True}


## 1. 평가 대상 기사 선택

후보 50행의 `article_idx`만 이용해 전처리 본문을 선택합니다. HCX 입력에는 정답 라벨 열을 넣지 않아 평가 누수를 막습니다.


In [26]:
articles = pd.read_csv(ARTICLE_PATH, encoding="utf-8-sig")
gold = pd.read_csv(GOLD_PATH, encoding="utf-8-sig")

articles = articles.reset_index().rename(columns={"index": "article_idx"})
articles["article_idx"] = articles["article_idx"].astype(int)
gold["article_idx"] = pd.to_numeric(gold["article_idx"], errors="raise").astype(int)

target_ids = gold["article_idx"].drop_duplicates().tolist()
target_articles = articles[articles["article_idx"].isin(target_ids)].copy()
target_articles = target_articles.sort_values("article_idx").reset_index(drop=True)

missing_ids = sorted(set(target_ids) - set(target_articles["article_idx"]))
assert not missing_ids, f"본문 파일에서 article_idx를 찾지 못함: {missing_ids}"
assert target_articles["본문_정제"].notna().all(), "본문_정제 결측치가 있습니다."

print(f"후보 행: {len(gold):,} / 고유 평가 기사: {len(target_articles):,} / 전체 기사: {len(articles):,}")
display(target_articles[["article_idx", "기사제목", "작성일", "본문_길이"]].head())


후보 행: 50 / 고유 평가 기사: 50 / 전체 기사: 2,706


,article_idx,기사제목,작성일,본문_길이
0,3,‘직원 한 명에 로봇 수십 대’… 이젠 로봇이 공장 움직인다,2025-01-01,2612
1,186,崔 대행 “취약부문별 맞춤형 일자리 지원 방안 마련하라”,2025-01-15,867
2,240,"작년 ‘꼴찌’ 韓 증시, 올해는 다르다… 연초 코스닥 수익률 6.86%",2025-01-19,4311
3,258,美 변압기공장 증설,2025-01-20,514
4,260,수도권 지하철 요금 상반기 중 150원 인상 전망,2025-01-21,528


## 2. Few-shot 프롬프트

평가 파일의 정답 문장을 예시로 재사용하지 않고, 경계가 분명한 합성 예시를 사용합니다. 핵심은 개별 기업 실적·계획·사건 수치와 집계통계를 구분하는 것입니다.


In [27]:
SYSTEM_PROMPT = r'''당신은 한국 뉴스의 수치 기반 주장을 구조화하는 데이터 분석가다.
기사에서 검증 가능한 집계통계 주장만 추출하고 KOSIS 검색 계획을 만든다.

[집계통계 포함]
- 인구, 고용, 물가, 무역, 복지, 교육 등 집단/기간을 집계한 관측 통계
- 평균, 비율, 증감, 순위, 규모 등 공식 통계표로 확인 가능한 주장

[제외]
- 개별 기업의 매출/투자/생산 계획, 주가/지분 거래
- 사건 1건, 개인 1명, 법령의 기준값, 정책 목표/전망/여론조사
- 행사 안내, 단순 날짜/가격/제품 사양

반드시 JSON 객체 하나만 출력한다. 설명이나 코드펜스를 쓰지 않는다.
스키마:
{"claims":[{
 "claim_text":"기사 원문에서 완결된 문장 그대로",
 "is_aggregate_claim":true,
 "claim_class":"집계통계",
 "indicator":"통계 지표명",
 "population":"대상 집단/지역",
 "value":"수치(복수면 ; 구분)",
 "unit":"단위(복수면 ; 구분)",
 "time_ref":"기준 시점",
 "time_compare":"비교 시점/대상",
 "source_org_mentioned":"기사에 명시된 기관 또는 빈 문자열",
 "expected_kosis_org":"KOSIS에서 예상되는 작성기관 또는 빈 문자열",
 "kosis_search_keywords":["짧고 구체적인 검색어 1","검색어 2"],
 "reason":"포함 근거"
}],"excluded":[{"text":"제외한 수치 문장","class":"개별사례|목표계획|전망예측|법령제도|여론조사|기타","reason":"짧은 이유"}]}

집계통계가 없으면 claims는 반드시 []로 둔다. 기관이나 통계표 ID를 지어내지 않는다.'''

FEW_SHOT = [
    ({"title": "취업자 증가", "text": "통계청에 따르면 지난달 취업자는 2,900만명으로 전년 동월보다 18만명 늘었다."},
     {"claims": [{"claim_text": "통계청에 따르면 지난달 취업자는 2,900만명으로 전년 동월보다 18만명 늘었다.", "is_aggregate_claim": True, "claim_class": "집계통계", "indicator": "취업자 수", "population": "전국 취업자", "value": "2900만;18만", "unit": "명;명", "time_ref": "지난달", "time_compare": "전년 동월", "source_org_mentioned": "통계청", "expected_kosis_org": "통계청", "kosis_search_keywords": ["취업자 수", "고용동향 취업자"], "reason": "전국 고용 집계의 시계열 수치"}], "excluded": []}),
    ({"title": "공장 증설", "text": "A사는 내년까지 4천억원을 투자해 생산량을 30% 늘릴 계획이다."},
     {"claims": [], "excluded": [{"text": "A사는 내년까지 4천억원을 투자해 생산량을 30% 늘릴 계획이다.", "class": "목표계획", "reason": "개별 기업의 미래 투자 계획"}]}),
    ({"title": "소비자물가", "text": "지난해 소비자물가는 1년 전보다 2.3% 올랐고 농축수산물은 5.9% 상승했다."},
     {"claims": [{"claim_text": "지난해 소비자물가는 1년 전보다 2.3% 올랐고 농축수산물은 5.9% 상승했다.", "is_aggregate_claim": True, "claim_class": "집계통계", "indicator": "소비자물가지수 상승률", "population": "전국", "value": "2.3;5.9", "unit": "%;%", "time_ref": "지난해", "time_compare": "1년 전", "source_org_mentioned": "", "expected_kosis_org": "통계청", "kosis_search_keywords": ["소비자물가지수", "농축수산물 소비자물가"], "reason": "공식 물가 집계로 확인 가능한 증감률"}], "excluded": []}),
]

def build_messages(row):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for x, y in FEW_SHOT:
        messages += [
            {"role": "user", "content": json.dumps(x, ensure_ascii=False)},
            {"role": "assistant", "content": json.dumps(y, ensure_ascii=False)},
        ]
    payload = {"article_idx": int(row.article_idx), "title": row.기사제목,
               "date": str(row.작성일), "text": row.본문_정제}
    messages.append({"role": "user", "content": json.dumps(payload, ensure_ascii=False)})
    return messages

print(build_messages(target_articles.iloc[0])[-1]["content"][:500])


{"article_idx": 3, "title": "‘직원 한 명에 로봇 수십 대’… 이젠 로봇이 공장 움직인다", "date": "2025-01-01", "text": "\"이곳에선 사람 한 명이 로봇 6대를 움직입니다. 로봇이 주요 공정의 100%를 처리하는 거죠.\" 지난달 중순 광주광역시 광산구 뉴서광 공장. 이곳에서 일하는 김형진 연구소장이 공장 안쪽을 가리키며 들려준 말이었다. 뉴서광은 냉장고와 같은 생활가전 전용문을 만드는 중소 제조업체다. 전체 공정의 70%를 다(多)관절 로봇을 활용해 자동화했다. 특히 주요 공정으로 꼽히는 철판 부품 삽입 및 조립과 문(door)을 프레스 공정을 거쳐 모양을 잡고 완성하는 과정에선 제어·관리하는 사람 한 명에 로봇 8대가 움직인다. 김 연구소장은 \"우리뿐 아니라 다른 제조업 공장을 가봐도 주요 공정은 사람 한 명에 로봇 여러 대가 붙어 처리한다\"며 \"로봇과의 협업은 이젠 일상\"이라고 했다. 우리나라는 1950~1960년대만 해


## 3. HCX 호출 및 안전한 JSON 파싱

`DRY_RUN=False`로 바꾸면 호출합니다. 성공 결과는 JSONL에 즉시 저장되므로 중단 후 다시 실행해도 완료된 기사는 건너뜁니다.


In [28]:
DRY_RUN = False

In [29]:
def extract_json_object(text: str) -> dict:
    text = text.strip().lstrip('\ufeff')
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text, flags=re.I)
    start, end = text.find("{"), text.rfind("}")
    if start < 0 or end <= start:
        raise ValueError(f"JSON 객체를 찾지 못함: {text[:200]}")
    candidate = text[start:end + 1]
    # 흔한 생성 오류(후행 쉼표, 제어문자)를 먼저 정리합니다.
    candidate = re.sub(r",\s*([}\]])", r"\1", candidate)
    candidate = ''.join(ch if ord(ch) >= 32 or ch in '\n\r\t' else ' ' for ch in candidate)
    try:
        return json.loads(candidate, strict=False)
    except json.JSONDecodeError as error:
        context = candidate[max(0, error.pos-80):error.pos+80]
        raise ValueError(
            f"HCX JSON 문법 오류({error.msg}, 위치 {error.pos}). 주변 내용: {context!r}"
        ) from error

def validate_result(obj: dict) -> dict:
    if not isinstance(obj, dict) or not isinstance(obj.get("claims"), list):
        raise ValueError("응답에 claims 배열이 없습니다.")
    obj.setdefault("excluded", [])
    for claim in obj["claims"]:
        if not claim.get("claim_text"):
            raise ValueError("claim_text가 비어 있습니다.")
        claim.setdefault("kosis_search_keywords", [])
    return obj

def hcx_chat(messages, retries=4):
    if not HCX_API_KEY:
        raise RuntimeError(".env에 NCP_CLOVASTUDIO_API_KEY가 필요합니다.")
    headers = {
        "Authorization": f"Bearer {HCX_API_KEY}",
        "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4()),
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    body = {"messages": messages, "temperature": 0.0, "topP": 0.8,
            "topK": 0, "maxTokens": 1800, "repetitionPenalty": 1.05, "stop": []}
    for attempt in range(retries):
        try:
            response = requests.post(HCX_URL, headers=headers, json=body, timeout=120)
            response.raise_for_status()
            data = response.json()
            content = (data.get("result", {}).get("message", {}).get("content")
                       or data.get("message", {}).get("content"))
            if isinstance(content, list):
                content = "".join(x.get("text", "") for x in content if isinstance(x, dict))
            if not content:
                raise ValueError(f"HCX 응답 본문 구조 확인 필요: {str(data)[:500]}")
            usage = data.get("result", {}).get("usage", data.get("usage", {}))
            try:
                parsed = extract_json_object(content)
            except ValueError as parse_error:
                # 따옴표 escape 등이 깨졌다면 모델에 JSON 재직렬화를 한 번 요청합니다.
                repair_body = {**body, "messages": [
                    {"role": "system", "content": (
                        "입력 내용을 변경·요약하지 말고 유효한 JSON 객체 하나로만 재직렬화하라. "
                        "문자열 내부의 큰따옴표와 줄바꿈을 반드시 JSON 규칙에 맞게 escape하라. "
                        "코드펜스와 설명은 출력하지 마라."
                    )},
                    {"role": "user", "content": content},
                ], "maxTokens": 2000}
                repair_headers = {**headers, "X-NCP-CLOVASTUDIO-REQUEST-ID": str(uuid.uuid4())}
                repaired_response = requests.post(
                    HCX_URL, headers=repair_headers, json=repair_body, timeout=120
                )
                repaired_response.raise_for_status()
                repaired_data = repaired_response.json()
                repaired_content = (
                    repaired_data.get("result", {}).get("message", {}).get("content")
                    or repaired_data.get("message", {}).get("content")
                )
                if isinstance(repaired_content, list):
                    repaired_content = "".join(
                        x.get("text", "") for x in repaired_content if isinstance(x, dict)
                    )
                if not repaired_content:
                    raise parse_error
                parsed = extract_json_object(repaired_content)
            return validate_result(parsed), usage
        except (requests.RequestException, ValueError, json.JSONDecodeError):
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)

def load_cache(path=CACHE_PATH):
    cache = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                item = json.loads(line)
                cache[int(item["article_idx"])] = item
    return cache

def append_cache(item, path=CACHE_PATH):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

def run_hcx(rows):
    cache = load_cache()
    work = rows.head(MAX_ARTICLES) if MAX_ARTICLES else rows
    for i, row in enumerate(work.itertuples(index=False), 1):
        idx = int(row.article_idx)
        if idx in cache:
            continue
        result, usage = hcx_chat(build_messages(row))
        item = {"article_idx": idx, "result": result, "usage": usage}
        append_cache(item)
        cache[idx] = item
        print(f"[{i}/{len(work)}] article_idx={idx}, claims={len(result['claims'])}")
    return cache

if DRY_RUN:
    print("DRY_RUN=True: 호출하지 않았습니다. 프롬프트를 확인한 뒤 False로 변경하세요.")
    hcx_cache = load_cache()
else:
    hcx_cache = run_hcx(target_articles)


## 4. KOSIS 공식 통계표 검색 및 추천

HCX가 만든 지표 검색어로 KOSIS `statisticsSearch.do`를 조회합니다. 반환된 실제 `ORG_ID`, `ORG_NM`, `TBL_ID`, `TBL_NM`만 결과에 기록합니다.


In [ ]:
def locate_kosis_metadata_file(filename: str) -> Path:
    candidates = [
        DATA_DIR / filename,
        DATA_DIR / "통계표" / filename,
        ROOT / filename,
        Path("/content/drive/MyDrive/news_verification/data") / filename,
        Path("/content/drive/MyDrive/news_verification/data/통계표") / filename,
        Path("/content") / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"{filename}을 찾지 못했습니다.\n" + "\n".join(map(str, candidates)))


KOSIS_TABLE_TREE_PATH = locate_kosis_metadata_file("kosis_table_tree.json")
KOSIS_ORG_NAMES_PATH = locate_kosis_metadata_file("kosis_org_names.json")
print("KOSIS table tree:", KOSIS_TABLE_TREE_PATH)
print("KOSIS org names:", KOSIS_ORG_NAMES_PATH)


def kosis_search(keyword: str, result_count=10) -> list[dict]:
    keyword = str(keyword).strip()
    if not keyword:
        return []
    if not KOSIS_API_KEY:
        raise RuntimeError(".env에 KOSIS_API_KEY가 필요합니다.")
    params = {
        "method": "getList", "apiKey": KOSIS_API_KEY, "searchNm": keyword,
        "sort": "RANK", "startCount": 1, "resultCount": result_count,
        "format": "json", "jsonVD": "Y",
    }
    response = requests.get(KOSIS_SEARCH_URL, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()
    if isinstance(data, dict):
        error_message = str(data.get("errMsg") or "").strip()
        if "데이터가 존재하지 않습니다" in error_message:
            return []
        raise RuntimeError(error_message or str(data))
    return data


def get_field(row, *names):
    for name in names:
        value = row.get(name) if isinstance(row, dict) else None
        if value not in (None, ""):
            return str(value)
    return ""


def token_set(text):
    return set(re.findall(r"[가-힣A-Za-z0-9]{2,}", str(text).lower()))


def load_local_kosis_catalog(tree_path=KOSIS_TABLE_TREE_PATH, org_path=KOSIS_ORG_NAMES_PATH):
    tree = json.loads(Path(tree_path).read_text(encoding="utf-8"))
    org_names = json.loads(Path(org_path).read_text(encoding="utf-8"))
    rows = []
    roots = tree.values() if isinstance(tree, dict) else tree
    for root in roots:
        if not isinstance(root, dict):
            continue
        for leaf in root.get("leaves", []):
            org_id = str(leaf.get("org_id", leaf.get("ORG_ID", "")))
            org_name = str(
                org_names.get(org_id, "") if isinstance(org_names, dict) else ""
            )
            rows.append({
                "ORG_ID": org_id,
                "ORG_NM": org_name,
                "TBL_ID": str(leaf.get("tbl_id", leaf.get("TBL_ID", ""))),
                "TBL_NM": str(leaf.get("tbl_nm", leaf.get("TBL_NM", ""))),
                "STAT_ID": str(leaf.get("stat_id", leaf.get("STAT_ID", ""))),
                "CATEGORY_PATH": " > ".join(map(str, leaf.get("path", []))),
                "candidate_source": "local_json",
            })
    catalog = pd.DataFrame(rows).drop_duplicates(["ORG_ID", "TBL_ID"]).reset_index(drop=True)
    catalog["SEARCH_TEXT"] = (
        catalog["ORG_NM"] + " " + catalog["TBL_NM"] + " " + catalog["CATEGORY_PATH"]
    ).str.lower()
    print(f"로컬 KOSIS 후보 {len(catalog):,}개 로드")
    return catalog, org_names


def search_local_kosis(query: str, catalog: pd.DataFrame, result_count=30) -> list[dict]:
    tokens = token_set(query) - {
        "통계", "자료", "관련", "기준", "전국", "대한", "따르면", "지난해", "올해"
    }
    if not tokens or catalog.empty:
        return []
    scores = pd.Series(0.0, index=catalog.index)
    for token in tokens:
        scores += catalog["SEARCH_TEXT"].str.contains(
            re.escape(token), regex=True, na=False
        ).astype(float) * (1.0 + min(len(token), 8) / 8)
    selected = scores[scores > 0].nlargest(result_count)
    result = catalog.loc[selected.index].drop(columns=["SEARCH_TEXT"]).copy()
    result["local_retrieval_score"] = selected.to_numpy()
    return result.to_dict("records")


def normalize_api_candidate(row: dict) -> dict:
    return {
        "ORG_ID": get_field(row, "ORG_ID", "orgId", "org_id"),
        "ORG_NM": get_field(row, "ORG_NM", "orgNm", "org_name"),
        "TBL_ID": get_field(row, "TBL_ID", "tblId", "tbl_id"),
        "TBL_NM": get_field(row, "TBL_NM", "tblNm", "tbl_name"),
        "STAT_ID": get_field(row, "STAT_ID", "statId", "stat_id"),
        "CATEGORY_PATH": get_field(row, "CATEGORY_PATH", "category_path"),
        "candidate_source": "kosis_api",
        "local_retrieval_score": 0.0,
    }


def merge_kosis_candidates(local_rows, api_rows, local_catalog):
    merged = {}
    catalog_by_table = {
        str(row.TBL_ID): row._asdict()
        for row in local_catalog.drop(columns=["SEARCH_TEXT"]).itertuples(index=False)
    }
    for raw in [*local_rows, *[normalize_api_candidate(row) for row in api_rows]]:
        table_id = get_field(raw, "TBL_ID")
        org_id = get_field(raw, "ORG_ID")
        key = (org_id, table_id) if org_id or table_id else None
        if key is None:
            continue
        local_match = catalog_by_table.get(table_id, {})
        candidate = {
            "ORG_ID": get_field(raw, "ORG_ID") or get_field(local_match, "ORG_ID"),
            "ORG_NM": get_field(raw, "ORG_NM") or get_field(local_match, "ORG_NM"),
            "TBL_ID": table_id,
            "TBL_NM": get_field(raw, "TBL_NM") or get_field(local_match, "TBL_NM"),
            "STAT_ID": get_field(raw, "STAT_ID") or get_field(local_match, "STAT_ID"),
            "CATEGORY_PATH": get_field(raw, "CATEGORY_PATH") or get_field(local_match, "CATEGORY_PATH"),
            "local_retrieval_score": float(raw.get("local_retrieval_score", 0) or 0),
            "from_local_json": raw.get("candidate_source") == "local_json",
            "from_kosis_api": raw.get("candidate_source") == "kosis_api",
        }
        if key in merged:
            previous = merged[key]
            for field in ["ORG_NM", "TBL_NM", "STAT_ID", "CATEGORY_PATH"]:
                previous[field] = previous[field] or candidate[field]
            previous["local_retrieval_score"] = max(
                previous["local_retrieval_score"], candidate["local_retrieval_score"]
            )
            previous["from_local_json"] |= candidate["from_local_json"]
            previous["from_kosis_api"] |= candidate["from_kosis_api"]
        else:
            merged[key] = candidate
    return list(merged.values())


def rank_kosis_candidates(claim, candidates):
    indicator = str(claim.get("indicator", ""))
    population = str(claim.get("population", ""))
    keywords = claim.get("kosis_search_keywords", []) or []
    expected_org = str(claim.get("expected_kosis_org", "")).strip()
    mentioned_org = str(claim.get("source_org_mentioned", "")).strip()
    query = " ".join([indicator, population, *map(str, keywords)]).strip()
    query_tokens = token_set(query)
    indicator_tokens = token_set(indicator)
    ranked = []
    for row in candidates:
        table_name = get_field(row, "TBL_NM")
        org_name = get_field(row, "ORG_NM")
        category_path = get_field(row, "CATEGORY_PATH")
        table_tokens = token_set(table_name)
        path_tokens = token_set(category_path)
        indicator_score = len(query_tokens & table_tokens) / max(1, len(query_tokens))
        if indicator and table_name:
            indicator_score = max(
                indicator_score,
                SequenceMatcher(None, indicator.lower(), table_name.lower()).ratio(),
            )
        org_targets = [value for value in [expected_org, mentioned_org] if value]
        org_score = max([
            1.0 if target in org_name or org_name in target else
            SequenceMatcher(None, target, org_name).ratio()
            for target in org_targets
        ], default=0.0)
        path_score = len((indicator_tokens or query_tokens) & path_tokens) / max(
            1, len(indicator_tokens or query_tokens)
        )
        api_score = 1.0 if row.get("from_kosis_api") else 0.0
        local_score = min(float(row.get("local_retrieval_score", 0)) / 10, 1.0)
        final_score = (
            0.50 * indicator_score + 0.25 * org_score
            + 0.15 * path_score + 0.05 * api_score + 0.05 * local_score
        )
        ranked.append({
            **row,
            "indicator_score": round(indicator_score, 4),
            "org_score": round(org_score, 4),
            "path_score": round(path_score, 4),
            "_score": round(final_score, 4),
        })
    return sorted(ranked, key=lambda row: row["_score"], reverse=True)


def recommend_kosis(cache):
    local_catalog, _ = load_local_kosis_catalog()
    output_rows = []
    for article_idx, item in cache.items():
        for claim_no, claim in enumerate(item["result"].get("claims", []), 1):
            keywords = [
                str(keyword).strip()
                for keyword in claim.get("kosis_search_keywords", [])[:3]
                if str(keyword).strip()
            ]
            combined_query = " ".join([
                str(claim.get("indicator", "")), str(claim.get("population", "")), *keywords
            ]).strip()
            local_rows = search_local_kosis(combined_query, local_catalog, result_count=40)
            api_rows = []
            for keyword in keywords:
                api_rows.extend(kosis_search(keyword, result_count=10))
                time.sleep(0.15)
            merged = merge_kosis_candidates(local_rows, api_rows, local_catalog)
            ranked = rank_kosis_candidates(claim, merged)[:TOP_K_KOSIS]
            if not ranked:
                output_rows.append({
                    "article_idx": article_idx, "claim_no": claim_no,
                    "claim_text": claim.get("claim_text", ""), "indicator": claim.get("indicator", ""),
                    "rank": "", "score": "", "org_id": "", "org_name": "",
                    "tbl_id": "", "tbl_name": "", "stat_id": "", "category_path": "",
                    "indicator_score": "", "org_score": "", "path_score": "",
                    "from_local_json": False, "from_kosis_api": False,
                })
                continue
            for rank, row in enumerate(ranked, 1):
                output_rows.append({
                    "article_idx": article_idx, "claim_no": claim_no,
                    "claim_text": claim.get("claim_text", ""), "indicator": claim.get("indicator", ""),
                    "rank": rank, "score": row["_score"],
                    "org_id": get_field(row, "ORG_ID"), "org_name": get_field(row, "ORG_NM"),
                    "tbl_id": get_field(row, "TBL_ID"), "tbl_name": get_field(row, "TBL_NM"),
                    "stat_id": get_field(row, "STAT_ID"),
                    "category_path": get_field(row, "CATEGORY_PATH"),
                    "indicator_score": row["indicator_score"], "org_score": row["org_score"],
                    "path_score": row["path_score"],
                    "from_local_json": row.get("from_local_json", False),
                    "from_kosis_api": row.get("from_kosis_api", False),
                })
    return pd.DataFrame(output_rows)


if DRY_RUN or not hcx_cache:
    recommendations = pd.DataFrame()
    print("HCX 캐시 결과가 생긴 뒤 KOSIS 추천을 실행합니다.")
else:
    recommendations = recommend_kosis(hcx_cache)
    recommendations.to_csv(RECOMMENDATION_PATH, index=False, encoding="utf-8-sig")
    display(recommendations.head(10))


KOSIS table tree: /content/drive/MyDrive/멋사/data/kosis_table_tree.json
KOSIS org names: /content/drive/MyDrive/멋사/data/kosis_org_names.json
로컬 KOSIS 후보 265,094개 로드


,article_idx,claim_no,claim_text,indicator,rank,score,org_id,org_name,tbl_id,tbl_name,stat_id,category_path,indicator_score,org_score,path_score,from_local_json,from_kosis_api
0,3,1,2023년 우리나라 로봇 밀도는 직원 1만명당 1012대였다.,로봇 밀도,1,0.3735,401,한국AI·로봇산업협회,TX_37302_A002_FRM373,로봇의 용도별 출하현황,2006203,광업ㆍ제조업 > 로봇산업실태조사,0.4706,0.3529,0.0,False,True
1,3,1,2023년 우리나라 로봇 밀도는 직원 1만명당 1012대였다.,로봇 밀도,2,0.3525,401,한국AI·로봇산업협회,DT_401N_001,로봇산업 생산현황,2006203,광업ㆍ제조업 > 로봇산업실태조사,0.4286,0.3529,0.0,False,True
2,3,1,2023년 우리나라 로봇 밀도는 직원 1만명당 1012대였다.,로봇 밀도,3,0.3525,401,한국AI·로봇산업협회,DT_401N_002,로봇산업 출하현황,2006203,광업ㆍ제조업 > 로봇산업실태조사,0.4286,0.3529,0.0,False,True
3,3,1,2023년 우리나라 로봇 밀도는 직원 1만명당 1012대였다.,로봇 밀도,4,0.3382,401,한국AI·로봇산업협회,DT_401N_014,로봇산업 출하 현황,2006203,광업ㆍ제조업 > 로봇산업실태조사,0.4000,0.3529,0.0,False,True
4,3,1,2023년 우리나라 로봇 밀도는 직원 1만명당 1012대였다.,로봇 밀도,5,0.3382,401,한국AI·로봇산업협회,DT_401N_013,로봇산업 생산 현황,2006203,광업ㆍ제조업 > 로봇산업실태조사,0.4000,0.3529,0.0,False,True
5,3,2,우리나라 로봇 밀도는 전 세계 평균(1만명당 162대)의 6배가 넘는다.,로봇 밀도 비교,1,0.2386,131,조달청,DT_13103_C025_OLD,일반용역 분류별 실적,2006071,정부ㆍ재정 > 조달통계 > 공공조달통계 > 업무대상 > 일반용역,0.2105,0.3333,0.0,False,True
6,3,2,우리나라 로봇 밀도는 전 세계 평균(1만명당 162대)의 6배가 넘는다.,로봇 밀도 비교,2,0.2082,338,한국여성정책연구원,DT_338001_2022138,현재 일자리의 교육수준 비교시 적합 만족도,2007337,사회일반 > 한국가족패널조사 > 2022년 > 일자리용,0.2581,0.2667,0.0,True,False
7,3,2,우리나라 로봇 밀도는 전 세계 평균(1만명당 162대)의 6배가 넘는다.,로봇 밀도 비교,3,0.1375,110,행정안전부,DT_110001_A001,성 및 연령별 인구와 인구밀도,1976004,사회일반 > 한국도시통계 > 한국도시통계(2023~) > 인구,0.2500,0.0000,0.0,True,False
8,3,2,우리나라 로봇 밀도는 전 세계 평균(1만명당 162대)의 6배가 넘는다.,로봇 밀도 비교,4,0.1375,110,행정안전부,DT_11001N_2013_A001,성 및 연령별 인구와 인구밀도,1976004,사회일반 > 한국도시통계 > 한국도시통계(2009~2022) > 인구,0.2500,0.0000,0.0,True,False
9,3,2,우리나라 로봇 밀도는 전 세계 평균(1만명당 162대)의 6배가 넘는다.,로봇 밀도 비교,5,0.1337,154,성평등가족부,DT_MOGE_3021100105,ISSP 국민정체성 항목별 국제지표 순위 비교,2015013,사회일반 > 국민다문화수용성조사 > 2018년 이전,0.2424,0.0000,0.0,True,False


## 5. 예측 평탄화 및 정답 매칭

동일 기사 안에서 gold claim과 예측 claim의 문자 정규화 유사도를 계산하고, 유사도가 가장 높은 쌍부터 1:1로 매칭합니다. 기본 임계값은 0.55이며 결과와 함께 저장합니다.


In [31]:
def normalize_text(text):
    return re.sub(r"[^0-9a-z가-힣]", "", str(text).lower())

def similarity(a, b):
    a, b = normalize_text(a), normalize_text(b)
    return SequenceMatcher(None, a, b).ratio() if a and b else 0.0

def flatten_predictions(cache):
    rows = []
    for article_idx, item in cache.items():
        for claim_no, claim in enumerate(item["result"]["claims"], 1):
            rows.append({"article_idx": article_idx, "pred_claim_no": claim_no, **claim})
    return pd.DataFrame(rows)

def greedy_match(gold_df, pred_df, threshold=0.55):
    pairs = []
    for article_idx, gpart in gold_df.groupby("article_idx"):
        ppart = pred_df[pred_df["article_idx"] == article_idx]
        candidates = [(similarity(g.claim_text, p.claim_text), gi, pi)
                      for gi, g in gpart.iterrows() for pi, p in ppart.iterrows()]
        used_g, used_p = set(), set()
        for score, gi, pi in sorted(candidates, reverse=True):
            if score < threshold or gi in used_g or pi in used_p:
                continue
            used_g.add(gi); used_p.add(pi)
            pairs.append({"gold_index": gi, "pred_index": pi, "text_similarity": score})
    return pd.DataFrame(pairs)

pred = flatten_predictions(hcx_cache)
matches = greedy_match(gold, pred) if not pred.empty else pd.DataFrame()
print(f"예측 claim {len(pred):,}개 / 매칭 {len(matches):,}개")


예측 claim 45개 / 매칭 3개


## 6. 성능 평가

- **Claim 추출**: 후보 파일에서 `gold_is_aggregate_claim=True`인 문장을 정답으로 보고 precision/recall/F1을 계산합니다.
- **분류**: 후보 50개 각각에 대해, 매칭된 HCX 집계통계 claim이 있으면 True로 간주합니다.
- **기관/표**: gold 기관 또는 지표가 실제로 채워진 행만 Top-k hit를 계산합니다. 미확정 라벨을 억지로 점수화하지 않습니다.


In [32]:
def as_bool(x):
    return str(x).strip().lower() in {"true", "1", "yes", "y"}

def safe_div(a, b):
    return a / b if b else 0.0

gold_eval = gold.copy()
gold_eval["gold_bool"] = gold_eval["gold_is_aggregate_claim"].map(as_bool)
matched_gold = set(matches["gold_index"].tolist()) if not matches.empty else set()
matched_pred = set(matches["pred_index"].tolist()) if not matches.empty else set()
positive_gold = set(gold_eval.index[gold_eval["gold_bool"]])

tp = len(matched_gold & positive_gold)
fp = len(pred) - len(matched_pred) + len(matched_gold - positive_gold)
fn = len(positive_gold - matched_gold)
precision, recall = safe_div(tp, tp + fp), safe_div(tp, tp + fn)
f1 = safe_div(2 * precision * recall, precision + recall)

gold_eval["pred_is_aggregate_claim"] = gold_eval.index.isin(matched_gold)
accuracy = (gold_eval["gold_bool"] == gold_eval["pred_is_aggregate_claim"]).mean()

metrics = pd.DataFrame([{
    "gold_positive": len(positive_gold), "predicted_claims": len(pred),
    "tp": tp, "fp": fp, "fn": fn,
    "precision": precision, "recall": recall, "f1": f1,
    "candidate_level_accuracy": accuracy,
}])
display(metrics.round(4))

# 비교 가능한 상세 결과 저장
detail = gold_eval.copy()
detail["matched"] = detail.index.isin(matched_gold)
detail["matched_pred_claim"] = ""
detail["text_similarity"] = 0.0
if not matches.empty:
    for m in matches.itertuples(index=False):
        detail.loc[m.gold_index, "matched_pred_claim"] = pred.loc[m.pred_index, "claim_text"]
        detail.loc[m.gold_index, "text_similarity"] = m.text_similarity
detail.to_csv(RESULT_PATH, index=False, encoding="utf-8-sig")

print("상세 비교:", RESULT_PATH)
if not recommendations.empty:
    print("KOSIS 추천:", RECOMMENDATION_PATH)


,gold_positive,predicted_claims,tp,fp,fn,precision,recall,f1,candidate_level_accuracy
0,20,45,2,43,18,0.0444,0.1,0.0615,0.62


상세 비교: /content/drive/MyDrive/멋사/output/fewshot/fewshot_hcx_predictions.csv
KOSIS 추천: /content/drive/MyDrive/멋사/output/fewshot/fewshot_kosis_recommendations.csv


In [33]:
# 기관/지표 정답이 존재하는 행만 선택적 평가
def is_filled(series):
    return series.fillna("").astype(str).str.strip().ne("")


gold_org_available = is_filled(
    gold_eval.get("gold_source_org_raw", pd.Series(index=gold_eval.index, dtype=str))
)
gold_indicator_available = is_filled(
    gold_eval.get("gold_indicator_raw", pd.Series(index=gold_eval.index, dtype=str))
)
scorable_mask = gold_org_available | gold_indicator_available
scorable = gold_eval[scorable_mask].copy()
not_scorable = gold_eval[~scorable_mask].copy()


def clean_text(value):
    if value is None or pd.isna(value):
        return ""
    return str(value).strip()


def compact_unique(values):
    result = []
    for value in values:
        text = str(value).strip()
        if text and text.lower() != "nan" and text not in result:
            result.append(text)
    return result


def build_kosis_evaluation_detail(scorable_rows, recommendation_rows):
    detail_rows = []
    for _, gold_row in scorable_rows.iterrows():
        article_key = str(gold_row.get("article_idx", ""))
        article_recommendations = recommendation_rows[
            recommendation_rows["article_idx"].astype(str).eq(article_key)
        ].copy() if not recommendation_rows.empty else pd.DataFrame()

        gold_org = clean_text(gold_row.get("gold_source_org_raw", ""))
        gold_indicator = clean_text(gold_row.get("gold_indicator_raw", ""))
        candidate_details = []
        org_hit = False
        indicator_hit = False
        maximum_similarity = 0.0

        for recommendation in article_recommendations.itertuples(index=False):
            predicted_org = clean_text(getattr(recommendation, "org_name", ""))
            predicted_table = clean_text(getattr(recommendation, "tbl_name", ""))
            predicted_table_id = clean_text(getattr(recommendation, "tbl_id", ""))
            current_org_hit = bool(gold_org and predicted_org and gold_org in predicted_org)
            current_similarity = similarity(gold_indicator, predicted_table) if gold_indicator and predicted_table else 0.0
            current_indicator_hit = bool(gold_indicator and current_similarity >= 0.45)
            org_hit = org_hit or current_org_hit
            indicator_hit = indicator_hit or current_indicator_hit
            maximum_similarity = max(maximum_similarity, current_similarity)
            candidate_details.append({
                "rank": getattr(recommendation, "rank", ""),
                "org_name": predicted_org,
                "tbl_id": predicted_table_id,
                "tbl_name": predicted_table,
                "org_hit": current_org_hit,
                "indicator_similarity": round(current_similarity, 4),
                "indicator_hit": current_indicator_hit,
            })

        valid_candidates = [
            candidate for candidate in candidate_details
            if candidate["org_name"] or candidate["tbl_id"] or candidate["tbl_name"]
        ]
        detail_rows.append({
            "sample_id": gold_row.get("sample_id", ""),
            "article_idx": gold_row.get("article_idx", ""),
            "기사제목": gold_row.get("기사제목", ""),
            "claim_text": gold_row.get("claim_text", ""),
            "gold_source_org_raw": gold_org,
            "gold_indicator_raw": gold_indicator,
            "evaluation_basis": "+".join([
                label for condition, label in [
                    (bool(gold_org), "기관명"), (bool(gold_indicator), "지표명")
                ] if condition
            ]),
            "recommendation_count": len(valid_candidates),
            "recommended_org_names": " | ".join(compact_unique(
                candidate["org_name"] for candidate in valid_candidates
            )),
            "recommended_table_ids": " | ".join(compact_unique(
                candidate["tbl_id"] for candidate in valid_candidates
            )),
            "recommended_table_names": " | ".join(compact_unique(
                candidate["tbl_name"] for candidate in valid_candidates
            )),
            "max_indicator_similarity": round(maximum_similarity, 4),
            "hit_by_org": org_hit,
            "hit_by_indicator": indicator_hit,
            "kosis_top_k_hit": org_hit or indicator_hit,
            "top_k_candidates_json": json.dumps(valid_candidates, ensure_ascii=False),
        })
    return pd.DataFrame(detail_rows)


if scorable.empty:
    kosis_top_k_detail = pd.DataFrame()
    print("기관명 또는 지표명 정답이 있는 gold 행이 없어 Top-k 평가를 생략합니다.")
else:
    kosis_top_k_detail = build_kosis_evaluation_detail(scorable, recommendations)
    hit_rate = float(kosis_top_k_detail["kosis_top_k_hit"].mean()) if len(kosis_top_k_detail) else 0.0
    print({
        "kosis_top_k_evaluable": len(kosis_top_k_detail),
        "kosis_top_k_hit_rate": hit_rate,
    })
    print("아래 행들이 실제 Top-k 평가 분모에 포함됐습니다.")
    display(kosis_top_k_detail[[
        "sample_id", "article_idx", "기사제목", "claim_text",
        "gold_source_org_raw", "gold_indicator_raw", "evaluation_basis",
        "recommendation_count", "recommended_org_names", "recommended_table_ids",
        "recommended_table_names", "max_indicator_similarity",
        "hit_by_org", "hit_by_indicator", "kosis_top_k_hit",
    ]])
    KOSIS_EVALUATION_DETAIL_PATH = OUTPUT_DIR / "fewshot_kosis_top_k_evaluation_detail.csv"
    kosis_top_k_detail.to_csv(KOSIS_EVALUATION_DETAIL_PATH, index=False, encoding="utf-8-sig")
    print("평가 상세 저장:", KOSIS_EVALUATION_DETAIL_PATH)

if not not_scorable.empty:
    print(f"기관명과 지표명 정답이 모두 없어 평가에서 제외된 gold 행: {len(not_scorable)}개")
    excluded_columns = [
        column for column in ["sample_id", "article_idx", "기사제목", "claim_text"]
        if column in not_scorable.columns
    ]
    display(not_scorable[excluded_columns].assign(exclusion_reason="gold 기관명·지표명 모두 없음"))


{'kosis_top_k_evaluable': 1, 'kosis_top_k_hit_rate': 0.0}
아래 행들이 실제 Top-k 평가 분모에 포함됐습니다.


,sample_id,article_idx,기사제목,claim_text,gold_source_org_raw,gold_indicator_raw,evaluation_basis,recommendation_count,recommended_org_names,recommended_table_ids,recommended_table_names,max_indicator_similarity,hit_by_org,hit_by_indicator,kosis_top_k_hit
0,C026,1381,기아 EV3 ‘세계 올해의 차’에...현대차그룹 4년 연속 수상,"시장조사업체 카이즈유데이터연구소에 따르면, 올 1분기(1~3월) 국내 전기차 판매 ...",카이즈유데이터연구소,,기관명,0,,,,0.0,False,False,False


평가 상세 저장: /content/drive/MyDrive/멋사/output/fewshot/fewshot_kosis_top_k_evaluation_detail.csv
기관명과 지표명 정답이 모두 없어 평가에서 제외된 gold 행: 49개


,sample_id,article_idx,기사제목,claim_text,exclusion_reason
0,C001,2362,"저소득층 많고 재정 열악한 지자체, 등골 더 휜다",중앙 부처 공무원들이 대부분인 세종은 기초생활수급자 비율이 0.4%로 전국에서 가장...,gold 기관명·지표명 모두 없음
1,C002,910,"고려아연 자회사 SMH, 영풍 지분 10.3% 확보… 영풍 의결권 제한",영풍은 이달 7일 의결권 제한을 받지 않기 위해 유한회사 와이피씨(YPC)를 신설해...,gold 기관명·지표명 모두 없음
2,C003,258,美 변압기공장 증설,"HD현대의 전력회사인 HD현대일렉트릭이 내년 초까지 약 4000억원을 투자해 울산,...",gold 기관명·지표명 모두 없음
3,C004,2179,"예대금리 차, 공시 이래 역대 최대 수준 유지",또 금리 인하 전인 작년 8월 4대 은행 예대금리 차는 0.23~0.71%포인트였는...,gold 기관명·지표명 모두 없음
4,C005,1731,주담대 연체율도 두 달 연속 최고치,1월(0.34%)에 이어 두 달 연속 최고치를 경신한 것이다.,gold 기관명·지표명 모두 없음
5,C006,480,이자도 못내는 ‘한계기업’ 8년새 2.7배 증가,"해당 기간 상승 폭은 12.3%포인트로, 미국(15.8%포인트)에 이어 둘째로 높았다.",gold 기관명·지표명 모두 없음
6,C007,2669,우리나라 거주 외국인 주민 '역대 최다' 258만명… 전체 인구의 5%,"집계 대상은 2024년 11월 1일 기준, 3개월을 초과해 국내에 장기 거주한 외국...",gold 기관명·지표명 모두 없음
7,C008,2622,"20대 인구, 70대보다도 적어졌다… 100년 만에 처음",한때 성인 가운데 가장 많았던 20대가 4년 연속 줄어들며 이제는 가장 적은 세대가...,gold 기관명·지표명 모두 없음
8,C009,2660,고령화가 바꾼 노동시장… 올해 상반기 일자리 증가 1위는 '돌봄·노인 일자리',"고령층을 돌보거나, 노인들을 위한 일자리 증가는 100만원 미만 저임금 일자리 확대...",gold 기관명·지표명 모두 없음
9,C010,2367,트럼프·머스크 갈등 심화...테슬라 주가 5.3% 하락,1일(현지 시각) 뉴욕증시에서 테슬라 주가는 전날보다 5.34% 내린 300.71달...,gold 기관명·지표명 모두 없음


## 실행 순서

1. `.env`에 `NCP_CLOVASTUDIO_API_KEY`, `KOSIS_API_KEY`가 있는지 확인합니다.
2. 설정 셀에서 `DRY_RUN=False`로 바꿉니다.
3. 처음에는 `MAX_ARTICLES=3`으로 전체 흐름을 시험합니다.
4. 정상이라면 `MAX_ARTICLES=None`으로 바꾸고 3번 셀부터 재실행합니다.
5. 결과는 `output/fewshot/` 아래 CSV 두 개와 JSONL 캐시에 저장됩니다.

API 사양 참고: [CLOVA Studio Chat Completions v3](https://api.ncloud-docs.com/docs/en/clovastudio-chatcompletionsv3), [KOSIS OpenAPI](https://kosis.kr/openapi/index/index.jsp)
